# 01. Obtención de datos

- **Fuente:** buscador de establecimientos educativos del MINEDUC, `https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/`.
- **Filtro aplicado:** `NIVEL ESCOLAR: DIVERSIFICADO`.
- **Cobertura:** los 22 departamentos oficiales de Guatemala. El buscador modela la ciudad capital como una entidad separada, `CIUDAD CAPITAL`, además de `GUATEMALA`.
- **Fecha de extracción:** 2026-07-20.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src import scraping

pd.set_option("display.max_columns", None)

## Cómo se obtuvo `establecimientos_raw.csv`

El sitio web de origen cuenta con medidas de seguridad que impidieron la descarga automatizada directa. Por esto se descargó manualmente los archivos de los 22 departamentos y Ciudad Capital desde el navegador. Finalmente, se utilizó el script `src/scraping.py` para unir todos estos archivos individuales en `establecimientos_raw.csv`.

In [2]:
RUTA_RAW = "../data/raw/establecimientos_raw.csv"

df_raw = pd.read_csv(RUTA_RAW, dtype=str)
print("Filas:", df_raw.shape[0], "Columnas:", df_raw.shape[1])
df_raw.head()

Filas: 11891 Columnas: 17


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,15-01-0050-46,15-01-0890,BAJA VERAPAZ,SALAMA,ESCUELA NORMAL RURAL NO.4 DR. ELIZARDO URIZAR ...,13 AVENIDA 7-41 ZONA 2 BARRIO HACIENDA DE LA V...,79400455,IRIS QUETZALI ORDOÑEZ TURCIOS,ANA ISABEL GUEVARA MEZA,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),BAJA VERAPAZ
1,15-01-0051-46,15-01-0890,BAJA VERAPAZ,SALAMA,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
2,15-01-0052-46,15-01-0895,BAJA VERAPAZ,SALAMA,LICEO MIXTO SAN MATEO,9A. AVENIDA 6-67 ZONA 1,79401926,NELSON JOEL HERNANDEZ HERNANDEZ,CARMEN ALICIA HERNÁNDEZ GUILLERMO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
3,15-01-0087-46,15-01-0890,BAJA VERAPAZ,SALAMA,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,"13 AVENIDA 7-41, ZONA 2, BARRIO HACIENDA DE LA...",53009229,IRIS QUETZALI ORDOÑEZ TURCIOS,ROSALINA NADEYDA TOJ MORENTE,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
4,15-01-0099-46,15-01-0890,BAJA VERAPAZ,SALAMA,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),BAJA VERAPAZ


## Cobertura por departamento

In [3]:
conteo_depto = df_raw["DEPARTAMENTO"].value_counts(dropna=False)
conteo_depto

DEPARTAMENTO
CIUDAD CAPITAL    2161
GUATEMALA         1908
SAN MARCOS         724
ESCUINTLA          708
HUEHUETENANGO      591
QUETZALTENANGO     551
PETEN              516
ALTA VERAPAZ       475
SUCHITEPEQUEZ      437
CHIMALTENANGO      435
SACATEPEQUEZ       430
IZABAL             413
JUTIAPA            392
RETALHULEU         364
QUICHE             323
CHIQUIMULA         239
SANTA ROSA         213
SOLOLA             192
JALAPA             183
BAJA VERAPAZ       171
EL PROGRESO        158
ZACAPA             156
TOTONICAPAN        128
NaN                 23
Name: count, dtype: int64

Se ven 23 categorías porque el combo del buscador MINEDUC separa
`CIUDAD CAPITAL`, código interno 00, de `GUATEMALA`, código 01. No es un
departamento 23 real, es la forma en que el buscador organiza la ciudad
capital. Los 22 departamentos oficiales de Guatemala sí están cubiertos. Esta
observación se retoma en el diagnóstico como un valor fuera de dominio a
resolver en la limpieza.

In [4]:
from src.catalogos import DEPARTAMENTOS_OFICIALES

departamentos_en_datos = set(df_raw["DEPARTAMENTO"].dropna().str.strip().str.upper())
faltantes = set(DEPARTAMENTOS_OFICIALES) - departamentos_en_datos
extra = departamentos_en_datos - set(DEPARTAMENTOS_OFICIALES)

print("Departamentos oficiales sin cubrir:", faltantes if faltantes else "ninguno")
print("Valores presentes que no son departamento oficial:", extra)

Departamentos oficiales sin cubrir: ninguno
Valores presentes que no son departamento oficial: {'CIUDAD CAPITAL'}
